In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")

from src.process_dataset_cnn import get_cnn_dataset
from src.models.CNN import CNN

# **Preprocesamiento y Alineación de Señales**

Antes de alimentar nuestra Red Neuronal Convolucional (CNN) 1D, es necesario transformar las señales crudas en un formato estructurado. Debido a que los sensores de frecuencia cardíaca (HR) y acelerometría operan a frecuencias de muestreo distintas, el paso crítico es sincronizar y alinear ambas fuentes con las etiquetas de las etapas del sueño.

### 1. Alineación Temporal mediante la Fórmula de Épocas

Siguiendo la documentación del dataset, cada época de sueño se define como un segmento de 30 segundos relativo al inicio del registro (recStart). Para asegurar que cada ventana de tiempo tenga asignada su etiqueta correcta, utilizamos la siguiente función de mapeo para cualquier marca de tiempo $t$:$$k = \lfloor \frac{t - recStart}{30} \rfloor + 1$$

Donde $k$ representa el índice de la época (ventana de 30 segundos) correspondiente. Este cálculo nos permite agrupar todas las lecturas de los sensores, sin importar su frecuencia original, dentro del segmento temporal exacto donde ocurrieron.

### 2. Homogeneización de Datos

Como la CNN requiere una entrada uniforme de dimensiones fijas, realizamos los siguientes ajustes:Muestreo Común: Llevamos ambas señales a una resolución de 1Hz (1 muestra por segundo) dentro de cada época de 30 segundos. Para la acelerometría (alta frecuencia), calculamos el promedio por segundo; para la frecuencia cardíaca, realizamos una interpolación lineal para rellenar los intervalos.Apilamiento (Stacking): Creamos una estructura de datos donde cada época de 30 segundos contiene 4 canales de información alineados: [HR, Acc_X, Acc_Y, Acc_Z].

### 3. Normalización

Dado que las magnitudes de la frecuencia cardíaca y la acelerometría difieren significativamente, aplicamos un StandardScaler. Esto asegura que todas las señales tengan media 0 y varianza 1, evitando que el modelo sesgue su aprendizaje hacia las variables con mayor valor numérico absoluto.

In [ ]:
X, y = get_cnn_dataset(n_patients=5)

Procesando pacientes:   0%|          | 0/5 [00:00<?, ?it/s]

# **Entrenamiento de una Red Neuronal Convolucional 1D**

Las señales biomédicas como la frecuencia cardíaca instantánea y la acelerometría poseen estructuras temporales locales (cambios bruscos, picos, periodos de reposo) que los filtros convolucionales pueden detectar eficientemente mediante operaciones de convolución 1D. Al aplicar estos filtros a lo largo del eje temporal, la red extrae automáticamente descriptores de alto nivel representativos de cada etapa del sueño, sin necesidad de realizar un feature engineering manual.

### Preparación de los datos

Para alimentar la red, los datos deben ser transformados en un tensor 3D:
-   Batch size: Número de épocas procesadas simultáneamente.
-   Steps: Longitud temporal de la secuencia (30 segundos).
-   Channels: Número de señales de entrada (IHR, Accel_X, Accel_Y, Accel_Z $\rightarrow$ 4 canales).

### Arquitectura Propuesta

La arquitectura se basa en bloques convolucionales que aprenden jerarquías de características locales, procesando las secuencias multicanal de manera más rápida que los modelos recurrentes. Este proceso emplea una estructura con capas de convolución, normalización y dropout para asegurar la estabilidad y mitigar el sobreajuste, culminando en un Global Average Pooling que condensa la información antes de la clasificación final. El aprendizaje se optimiza mediante Categorical Crossentropy y mecanismos de early stopping, permitiendo mapear eficazmente la dinámica local de las señales a las etapas del sueño correspondientes.